In [8]:
import pandas as pd
import ast
from rich.progress import track
import nltk
import Stemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import linear_model
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import swifter
import numpy as np

In [21]:
all_columns = [
    'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at', 
    'updated_at', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']
converters_dict = {col: ast.literal_eval for col in all_columns}

training_set = pd.read_csv('../Group_project/training_set.csv', converters = converters_dict)

In [22]:
full_voc = pd.read_csv('../Group_project/vocabulary_995000.csv')
sub_voc = set((full_voc.iloc[:10000])['word'].values.tolist())
print(sub_voc)

{'effect', 'status', 'vandal', 'mass', 'px', 'graham', 'femal', 'racket', 'typo', 'rigid', 'wendi', 'intim', 'ussr', 'wrongdo', 'conjur', 'sorrow', 'maher', 'reprint', 'bonanza', 'envi', 'fragment', 'reserv', 'aston', 'even', 'rye', 'educ', 'robe', 'strong', 'shoot', 'perimet', 'abus', 'fm', 'cato', 'queer', 'withdrawn', 'carbon', 'tim', 'falsehood', 'wait', 'repertori', 'harriet', 'govern', 'cycl', 'tract', 'scare', 'adel', 'mcmaster', 'furnitur', 'recsa', 'tangibl', 'celtic', 'dprk', 'expedit', 'subprim', 'jacob', 'debacl', 'gas', 'ironi', 'jupit', 'fought', 'taxi', 'cement', 'medicar', 'natali', 'upstat', 'upheld', 'osama', 'takeaway', 'sophist', 'southern', 'mat', 'insur', 'becker', 'shovel', 'monarchi', 'intric', 'swamp', 'cnbc', 'cardiovascular', 'coincid', 'memori', 'moscow', 'cherri', 'boy', 'nullifi', 'uncomfort', 'ring', 'rap', 'actor', 'comparison', 'roger', 'gowdi', 'complicit', 'councilman', 'south', 'levant', 'fundament', 'incompet', 'depress', 'heavi', 'disastr', 'shortc

In [23]:
types = {i[0] for i in training_set['type'] if len(i) < 2}
print(types)
fake_news = {'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
reliable_news = {'reliabl', 'unreli', 'polit', 'clickbait'}

fake = {'false', 'half-true', 'pants-fire'}
reliable = {'barely-true', 'mostly-true', 'true'}

def filter_type(type, fake, rel):
    if type in fake:
        return 'fake'
    elif type in rel:
        return 'reliable'

{'hate', 'unreli', 'fake', 'bias', 'rumor', 'clickbait', 'unknown', 'junksci', 'satir', 'polit', 'reliabl', 'conspiraci'}


In [24]:
def filter_strings(str_list, **kwargs):
    filtered = [word for word in str_list if word in sub_voc]
    joined = ' '.join(filtered)
    return joined

only_content = pd.DataFrame({})
only_content['content'] = training_set['content'].swifter.progress_bar(False).apply(filter_strings)
only_content['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

x_oc = only_content[only_content['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_oc = x_oc['y_t']

vectorizer_oc = CountVectorizer()
X_oc = vectorizer_oc.fit_transform(x_oc['content'])
Y_oc = np.array(y_oc)

In [25]:
logr = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr.fit(X_oc, Y_oc)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [36]:
liar_set = pd.read_csv('../Group_project/liar_dataset/train.tsv', sep = '\t', dtype = 'string')
test_set = pd.read_csv('../Group_project/test_set.csv', converters = converters_dict)

In [37]:
test = pd.DataFrame({})
test['content'] = test_set['content'].swifter.progress_bar(False).apply(filter_strings)
test['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

x_oc_t = test[test['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
oc_y = x_oc_t['y_t']

X_test = vectorizer_oc.transform(x_oc_t['content'])
Y_test = np.array(oc_y)

In [27]:
w_stemmer = Stemmer.Stemmer('english')

liar_data = pd.DataFrame({})
liar_data['tokenized'] = liar_set['Statement'].swifter.progress_bar(False).apply(
            lambda x, **kwargs: nltk.word_tokenize(str(x)))
liar_data['stemmed'] = liar_data['tokenized'].swifter.progress_bar(False).apply(
            lambda x, **kwargs: w_stemmer.stemWords(x))


In [29]:
liar_data['statement'] = liar_data['stemmed'].swifter.progress_bar(False).apply(lambda x, **kwargs: ' '.join([word for word in x if word in sub_voc]))
# print(liar_data['statement'])

liar_data['y_t'] = liar_set['Label'].apply(lambda x, **kwargs: filter_type(x, fake, reliable))
print(liar_data['y_t'])

0            fake
1            fake
2        reliable
3            fake
4            fake
           ...   
10235    reliable
10236    reliable
10237        fake
10238        fake
10239        fake
Name: y_t, Length: 10240, dtype: str


In [39]:
X_liar = vectorizer_oc.transform(liar_data['statement'])
y_liar = np.array(liar_data['y_t'])

test_predictions = logr.predict(X_test)

liar_predictions = logr.predict(X_liar)

In [40]:
print(f'The acuracy score for the model is: {accuracy_score(Y_test, test_predictions)}')
print('The confusion matrix for the model:')
print(confusion_matrix(Y_test, test_predictions))
print('The classification report of the logistic model is:')
print(classification_report(Y_test, test_predictions))

The acuracy score for the model is: 0.8493961218836565
The confusion matrix for the model:
[[37706  5035]
 [ 8557 38952]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.82      0.88      0.85     42741
    reliable       0.89      0.82      0.85     47509

    accuracy                           0.85     90250
   macro avg       0.85      0.85      0.85     90250
weighted avg       0.85      0.85      0.85     90250



In [42]:
print(f'The acuracy score for the model is: {accuracy_score(y_liar, liar_predictions)}')
print('The confusion matrix for the model:')
print(confusion_matrix(y_liar, liar_predictions))
print('The classification report of the logistic model is:')
print(classification_report(y_liar, liar_predictions))

The acuracy score for the model is: 0.4826171875
The confusion matrix for the model:
[[4811  137]
 [5161  131]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.48      0.97      0.64      4948
    reliable       0.49      0.02      0.05      5292

    accuracy                           0.48     10240
   macro avg       0.49      0.50      0.35     10240
weighted avg       0.49      0.48      0.34     10240

